# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aslam-khalid/Flyrank-Week-1/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Before running the data loading cell, we need to ensure the `Flyrank-Week-1` repository is available and that the notebook's working directory is set correctly to resolve the relative path to the data file. The error indicates that the file `../../data/raw/content_refresh_anonymized.csv` was not found, which is typical if the repository hasn't been cloned or the current directory is not within `Flyrank-Week-1/work/notebooks`.

In [7]:
# Clone the Flyrank-Week-1 repository
!git clone https://github.com/aslam-khalid/Flyrank-Week-1


Cloning into 'Flyrank-Week-1'...
remote: Enumerating objects: 143, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 143 (delta 56), reused 110 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (143/143), 1.85 MiB | 7.70 MiB/s, done.
Resolving deltas: 100% (56/56), done.


In [8]:
# Change the current working directory to the notebook's expected location within the repository
# This assumes the notebook is meant to be run from 'Flyrank-Week-1/work/notebooks'
import os

# Navigate to the root of the cloned repository first
if not os.path.exists('Flyrank-Week-1'):
    print("Repository not found. Please ensure it was cloned successfully.")
else:
    # Change to the directory where the notebook typically resides in the repo
    os.chdir('Flyrank-Week-1/work/notebooks')
    print(f"Current working directory changed to: {os.getcwd()}")

# Verify the path to the data file relative to the new CWD
# The path '../../data/raw/content_refresh_anonymized.csv' should now resolve correctly
# to /content/Flyrank-Week-1/data/raw/content_refresh_anonymized.csv


Current working directory changed to: /content/Flyrank-Week-1/work/notebooks/Flyrank-Week-1/work/notebooks


After executing the above cells, the data loading cell (cell `Mon7V-EIYbKt`) should now be able to find and load the `content_refresh_anonymized.csv` file.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.





# no numbers needed here — see Section 3


###Lane: Refresh / Content Opportunity Scoring.

This lane asks which pages should be reviewed first for refresh, expansion, protection, pruning, or
monitoring. I'm picking it over the other three for three reasons. First, it's the lane the starter
pipeline (scripts/01-05) already builds end-to-end, so I can check real, verified results before
committing 7 weeks to it — see the numbers below. Second, it's a ranking/priority problem under
limited review capacity, the same shape as my FloodSense project (XGBoost binary + severity scoring
on an imbalanced panel with SMOTE) — precision@K, thresholds, and reason codes all transfer directly.
Third, it produces one concrete, defensible artifact (a ranked queue with reason codes) instead of an
open-ended correlation report, keeping the 7 weeks anchored to a single deliverable I can keep
improving rather than re-scoping every week.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Decision: given a limited weekly review capacity, which content pages should a content editor look at
first — and with what suggested action (refresh, expand, protect, prune, or monitor)?

Who acts, what they do: a content editor opens the ranked queue, reads the reason codes on the top N
pages, and either refreshes the page (updates copy, expands thin sections, rewrites metadata) or
moves to the next candidate. The unit of analysis is one content item (one row = one pseudonymized
page, trailing-90-day window).

Cost of a wrong call:
- False positive (flagged, actually fine): wastes scarce editor hours, and the opportunity cost is a
  real declining page that didn't get reviewed instead.
- False negative (missed a real decline/opportunity): the page keeps losing visibility or clicks
  silently, which compounds over time — this is the more expensive error for a client whose organic
  traffic pays the bills, so precision@K (matched to real review capacity) is the right metric, not
  raw accuracy.

Why data/ML, not just a rule: Section 3 shows the existing rule-based baseline is measurably worse
than a learned ranking on this same data — the pattern is real but spread across several tangled,
weak signals (freshness, position, demand, engagement) that one hand-written if-statement can't
combine well.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# no numbers needed here — see Section 3




## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Starter dataset: {df.shape[0]:,} rows x {df.shape[1]} columns, "
      f"{df['client_id'].nunique()} clients")

declining = (df["trend_direction"] == "down").mean() * 100
print(f"\n1) {declining:.1f}% of pages are trend_direction == 'down' "
      f"({(df['trend_direction']=='down').sum():,} rows) -- too many for manual review, "
      f"which is exactly the prioritization problem this lane solves.")

baseline_p50, rf_p50 = 0.240, 0.740
print(f"\n2) Starter pipeline's rule baseline gets precision@50 = {baseline_p50} "
      f"(~{int(baseline_p50*50)}/50 correct) vs random forest precision@50 = {rf_p50} "
      f"(~{int(rf_p50*50)}/50 correct) -- a learned rank beats the hand-written rule.")

visible_and_declining = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
print(f"\n3) {len(visible_and_declining):,} pages are declining AND have real search demand "
      f"(impressions_90d >= 100) -- a large candidate pool worth ranking, not noise.")


Starter dataset: 30,000 rows x 44 columns, 32 clients

1) 54.2% of pages are trend_direction == 'down' (16,262 rows) -- too many for manual review, which is exactly the prioritization problem this lane solves.

2) Starter pipeline's rule baseline gets precision@50 = 0.24 (~12/50 correct) vs random forest precision@50 = 0.74 (~37/50 correct) -- a learned rank beats the hand-written rule.

3) 13,152 pages are declining AND have real search demand (impressions_90d >= 100) -- a large candidate pool worth ranking, not noise.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can claim: observed associations between content/search signals and the outcome I define on
this snapshot; directional, decision-support recommendations ("worth reviewing first, given the
evidence") — not guarantees; a rule-baseline vs. learned-model comparison using precision@K matched
to real review capacity.

What I will not claim: that a refresh causes a recovery — that needs an experiment, not this
observational data; anything about Google's ranking algorithm or AI citations/rankings; that
trend_direction (the starter's proxy label, built from the current window) is the ideal target — it's
a beginner proxy, and a stronger version of this lane will define the label from a future outcome
window (prior 90 days of features -> decline/recovery over the next 30 days); that any single flagged
page is a guaranteed real decline rather than consolidation, seasonality, or noise — I'll rule those
out before trusting a flag.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# no numbers needed here — see Section 3


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.